# LESSON 4.4: Image Smoothing Using Lowpass Frequency Domain Filters
## Filtering in the Frequency Domain

In this lesson:
- Why lowpass filtering smooths images
- Ideal Lowpass Filter (ILPF) and ringing artifacts
- Gaussian Lowpass Filter (GLPF) and its smooth transition
- Butterworth Lowpass Filter (BLPF) and the role of filter order
- Comparison of all three filters
- Effect of cutoff frequency $D_0$ on the filtered result
- Power spectrum analysis and enclosed energy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Introduction: Frequency Content of Images

In the frequency domain representation of an image:

- **Low frequencies** correspond to **smooth, slowly varying** regions (large-scale intensity patterns)
- **High frequencies** correspond to **edges, fine details, and noise** (rapid intensity changes)

### Lowpass Filtering

A **lowpass filter** attenuates high-frequency components while passing low-frequency components. The result is a **smoothed (blurred)** image.

### Frequency Domain Filtering Steps (Gonzalez, Section 4.7):

1. Compute the DFT: $F(u,v) = \mathcal{F}\{f(x,y)\}$
2. Center the spectrum: shift zero-frequency to the middle
3. Multiply by the filter: $G(u,v) = H(u,v) \cdot F(u,v)$
4. Shift back and compute inverse DFT: $g(x,y) = \mathcal{F}^{-1}\{G(u,v)\}$

### Distance from Center

All lowpass filters are defined using the distance from each point $(u,v)$ to the center of the frequency rectangle:

$$D(u,v) = \sqrt{\left(u - \frac{P}{2}\right)^2 + \left(v - \frac{Q}{2}\right)^2}$$

where $P \times Q$ is the size of the (zero-padded) image.

In [ ]:
# Helper functions used throughout this notebook

def create_distance_matrix(P, Q):
    """
    Create the distance matrix D(u,v) from center of frequency rectangle.
    D(u,v) = sqrt((u - P/2)^2 + (v - Q/2)^2)
    """
    u = np.arange(P)
    v = np.arange(Q)
    U, V = np.meshgrid(u, v, indexing='ij')
    D = np.sqrt((U - P / 2) ** 2 + (V - Q / 2) ** 2)
    return D


def apply_frequency_filter(image, H):
    """
    Apply a frequency domain filter H to an image.
    
    Steps:
    1. Compute centered DFT of the image
    2. Multiply by filter H(u,v)
    3. Inverse DFT and take real part
    
    Parameters:
        image: 2D numpy array (grayscale)
        H: 2D filter in the frequency domain (same size as image)
    Returns:
        Filtered image (real, clipped to [0, 255])
    """
    F = np.fft.fftshift(np.fft.fft2(image))
    G = H * F
    g = np.real(np.fft.ifft2(np.fft.ifftshift(G)))
    g = np.clip(g, 0, 255)
    return g


def create_test_image(size=256):
    """
    Create a synthetic test image with edges, smooth regions, and fine detail.
    Simulates a simplified biomedical image.
    """
    img = np.zeros((size, size), dtype=np.float64)
    
    # Background: smooth gradient
    Y, X = np.mgrid[0:size, 0:size]
    img += 40 + 20 * np.sin(2 * np.pi * X / size)
    
    # Large bright ellipse (simulating an organ or tissue region)
    cx, cy = size // 2, size // 2
    mask_ellipse = ((X - cx) / 70) ** 2 + ((Y - cy) / 50) ** 2 <= 1
    img[mask_ellipse] = 180
    
    # Smaller bright circle inside
    mask_circle = (X - cx + 15) ** 2 + (Y - cy - 10) ** 2 <= 20 ** 2
    img[mask_circle] = 220
    
    # Small dark circle (simulating a lesion)
    mask_lesion = (X - cx - 25) ** 2 + (Y - cy + 15) ** 2 <= 10 ** 2
    img[mask_lesion] = 60
    
    # Sharp rectangular feature
    img[30:50, 30:100] = 200
    
    # Fine grid pattern (high frequency detail)
    for i in range(180, 240, 4):
        img[i, 30:100] = 200
    for j in range(30, 100, 4):
        img[180:240, j] = 200
    
    # Add some noise
    np.random.seed(42)
    img += np.random.normal(0, 5, img.shape)
    img = np.clip(img, 0, 255)
    
    return img


# Create the test image
test_img = create_test_image(256)
P, Q = test_img.shape

plt.figure(figsize=(6, 6))
plt.imshow(test_img, cmap='gray', vmin=0, vmax=255)
plt.title('Test Image (256 x 256)', fontsize=13)
plt.colorbar()
plt.show()

In [ ]:
# Visualize the frequency content of the test image
F = np.fft.fftshift(np.fft.fft2(test_img))
spectrum = np.log1p(np.abs(F))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(test_img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Spatial Domain: f(x,y)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(spectrum, cmap='gray')
axes[1].set_title('Frequency Domain: log(1 + |F(u,v)|)', fontsize=12)
axes[1].axis('off')

plt.suptitle('Test Image and Its Fourier Spectrum', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Bright center = low frequencies (smooth regions)")
print("Spread-out energy = high frequencies (edges, details, noise)")

---
## 2. Ideal Lowpass Filter (ILPF)

The **Ideal Lowpass Filter** has a sharp cutoff at distance $D_0$:

$$H(u,v) = \begin{cases} 1 & \text{if } D(u,v) \leq D_0 \\ 0 & \text{if } D(u,v) > D_0 \end{cases}$$

Where:
- $D_0$ = **cutoff frequency** (radius of the passband circle)
- $D(u,v) = \sqrt{(u - P/2)^2 + (v - Q/2)^2}$

### Characteristics:
- **Perfect** in theory: passes all frequencies below $D_0$ and blocks all above
- In practice, the **sharp transition** causes severe **ringing artifacts** (Gibbs phenomenon)
- The impulse response (inverse DFT of the ILPF) is a **sinc function**, which has infinite oscillations
- **Not used in practice** due to ringing, but serves as a theoretical reference

In [ ]:
def ideal_lowpass_filter(P, Q, D0):
    """
    Create an Ideal Lowpass Filter.
    H(u,v) = 1 if D(u,v) <= D0, else 0
    """
    D = create_distance_matrix(P, Q)
    H = np.zeros((P, Q), dtype=np.float64)
    H[D <= D0] = 1.0
    return H


# Show the ILPF for different cutoff frequencies
D0_values = [10, 30, 60, 100]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, D0 in enumerate(D0_values):
    H = ideal_lowpass_filter(P, Q, D0)
    axes[i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f'ILPF, $D_0$ = {D0}', fontsize=12)
    axes[i].axis('off')

plt.suptitle('Ideal Lowpass Filter H(u,v) for Different Cutoff Frequencies',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Radial cross-section of the ILPF
D = create_distance_matrix(P, Q)
center_u, center_v = P // 2, Q // 2
radial_distances = D[center_u, center_v:]  # horizontal profile from center

fig, ax = plt.subplots(figsize=(8, 5))

for D0 in [10, 30, 60, 100]:
    H = ideal_lowpass_filter(P, Q, D0)
    profile = H[center_u, center_v:]  # radial profile
    ax.plot(radial_distances, profile, linewidth=2, label=f'$D_0$ = {D0}')

ax.set_xlabel('Distance D(u,v) from Center', fontsize=12)
ax.set_ylabel('H(u,v)', fontsize=12)
ax.set_title('ILPF Radial Cross-Section: Sharp Transition', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 128])
ax.set_ylim([-0.05, 1.1])
plt.tight_layout()
plt.show()

In [ ]:
# Apply ILPF and show ringing effect
D0_values = [10, 30, 60]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, D0 in enumerate(D0_values):
    H = ideal_lowpass_filter(P, Q, D0)
    filtered = apply_frequency_filter(test_img, H)
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'ILPF: $D_0$ = {D0}', fontsize=12)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(filtered, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Filtered Result ($D_0$ = {D0})', fontsize=12)
    axes[1, i].axis('off')

plt.suptitle('Ideal Lowpass Filter: Notice the Ringing Artifacts!\n'
             'Ringing is most visible with small $D_0$ (severe blurring)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("RINGING EFFECT: The sharp cutoff of the ILPF creates oscillations")
print("(ripple patterns) around edges. This is the Gibbs phenomenon.")
print("Smaller D0 = more blurring AND more visible ringing.")

In [ ]:
# Zoom into a region to clearly show ringing
H_10 = ideal_lowpass_filter(P, Q, 10)
filtered_10 = apply_frequency_filter(test_img, H_10)

# Extract a region around the rectangular feature
row_start, row_end = 10, 80
col_start, col_end = 10, 120

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(test_img[row_start:row_end, col_start:col_end], cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original (zoomed)', fontsize=12)

axes[1].imshow(filtered_10[row_start:row_end, col_start:col_end], cmap='gray', vmin=0, vmax=255)
axes[1].set_title('ILPF $D_0$=10 (zoomed) - Ringing Visible', fontsize=12)

plt.suptitle('Ringing Artifact Close-Up', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Also show intensity profile across an edge
row_idx = 40
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test_img[row_idx, :], 'b-', linewidth=1.5, label='Original')
ax.plot(filtered_10[row_idx, :], 'r-', linewidth=2, label='ILPF $D_0$=10')
ax.set_xlabel('Column Index', fontsize=12)
ax.set_ylabel('Pixel Intensity', fontsize=12)
ax.set_title(f'Intensity Profile Along Row {row_idx}: Ringing Oscillations', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Gaussian Lowpass Filter (GLPF)

The **Gaussian Lowpass Filter** has a smooth, bell-shaped transfer function:

$$H(u,v) = e^{-D^2(u,v) \,/\, 2D_0^2}$$

Where:
- $D_0$ = cutoff frequency (distance at which $H$ drops to $e^{-1/2} \approx 0.607$)
- $D(u,v) = \sqrt{(u - P/2)^2 + (v - Q/2)^2}$

### Key Properties:
- The inverse DFT of a Gaussian is **also a Gaussian** (unique self-dual property)
- **No ringing artifacts** because the smooth transition produces no oscillations
- At $D(u,v) = D_0$: $H = e^{-0.5} \approx 0.607$
- The GLPF achieves the **minimum possible spread** in both spatial and frequency domains simultaneously (uncertainty principle)
- Widely used in biomedical imaging due to artifact-free smoothing

In [ ]:
def gaussian_lowpass_filter(P, Q, D0):
    """
    Create a Gaussian Lowpass Filter.
    H(u,v) = exp(-D(u,v)^2 / (2 * D0^2))
    """
    D = create_distance_matrix(P, Q)
    H = np.exp(-(D ** 2) / (2 * D0 ** 2))
    return H


# Show the GLPF for different cutoff frequencies
D0_values = [10, 30, 60, 100]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, D0 in enumerate(D0_values):
    H = gaussian_lowpass_filter(P, Q, D0)
    axes[i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f'GLPF, $D_0$ = {D0}', fontsize=12)
    axes[i].axis('off')

plt.suptitle('Gaussian Lowpass Filter H(u,v) for Different Cutoff Frequencies',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Radial cross-section of the GLPF
D = create_distance_matrix(P, Q)
center_u, center_v = P // 2, Q // 2
radial_distances = D[center_u, center_v:]

fig, ax = plt.subplots(figsize=(8, 5))

for D0 in [10, 30, 60, 100]:
    H = gaussian_lowpass_filter(P, Q, D0)
    profile = H[center_u, center_v:]
    ax.plot(radial_distances, profile, linewidth=2, label=f'$D_0$ = {D0}')

# Mark the 0.607 level
ax.axhline(y=np.exp(-0.5), color='gray', linestyle='--', alpha=0.7,
           label=f'$e^{{-1/2}}$ = {np.exp(-0.5):.3f}')

ax.set_xlabel('Distance D(u,v) from Center', fontsize=12)
ax.set_ylabel('H(u,v)', fontsize=12)
ax.set_title('GLPF Radial Cross-Section: Smooth Gaussian Transition', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 128])
ax.set_ylim([-0.05, 1.1])
plt.tight_layout()
plt.show()

In [ ]:
# Apply GLPF and compare with ILPF: no ringing!
D0_values = [10, 30, 60]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, D0 in enumerate(D0_values):
    H = gaussian_lowpass_filter(P, Q, D0)
    filtered = apply_frequency_filter(test_img, H)
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'GLPF: $D_0$ = {D0}', fontsize=12)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(filtered, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Filtered Result ($D_0$ = {D0})', fontsize=12)
    axes[1, i].axis('off')

plt.suptitle('Gaussian Lowpass Filter: Smooth Blurring, No Ringing!',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key observation: GLPF produces smooth blurring WITHOUT ringing artifacts.")
print("This is because the Gaussian has no sharp transitions in the frequency domain.")

In [ ]:
# Compare ILPF vs GLPF intensity profiles to highlight ringing difference
H_ilpf = ideal_lowpass_filter(P, Q, 15)
H_glpf = gaussian_lowpass_filter(P, Q, 15)

filtered_ilpf = apply_frequency_filter(test_img, H_ilpf)
filtered_glpf = apply_frequency_filter(test_img, H_glpf)

row_idx = 40

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test_img[row_idx, :], 'k-', linewidth=1, alpha=0.5, label='Original')
ax.plot(filtered_ilpf[row_idx, :], 'r-', linewidth=2, label='ILPF $D_0$=15 (ringing)')
ax.plot(filtered_glpf[row_idx, :], 'b-', linewidth=2, label='GLPF $D_0$=15 (smooth)')
ax.set_xlabel('Column Index', fontsize=12)
ax.set_ylabel('Pixel Intensity', fontsize=12)
ax.set_title(f'Row {row_idx} Profile: ILPF Ringing vs GLPF Smooth Transition', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Butterworth Lowpass Filter (BLPF)

The **Butterworth Lowpass Filter** of order $n$:

$$H(u,v) = \frac{1}{1 + \left[\dfrac{D(u,v)}{D_0}\right]^{2n}}$$

Where:
- $D_0$ = cutoff frequency (distance at which $H = 0.5$, i.e., $-3\text{ dB}$ point)
- $n$ = filter order, controls the sharpness of the transition

### Key Properties:
- At $D(u,v) = D_0$: $H = \frac{1}{1+1} = 0.5$ (always, for any $n$)
- **Low order** ($n=1, 2$): smooth transition, similar to Gaussian, **no ringing**
- **High order** ($n \to \infty$): approaches the Ideal filter, **ringing appears**
- Butterworth provides a **tunable transition** between Gaussian-like and Ideal-like behavior
- Unlike ILPF, the transfer function never has a true discontinuity

In [ ]:
def butterworth_lowpass_filter(P, Q, D0, n):
    """
    Create a Butterworth Lowpass Filter of order n.
    H(u,v) = 1 / (1 + (D(u,v)/D0)^(2n))
    """
    D = create_distance_matrix(P, Q)
    # Avoid division by zero at center
    H = 1.0 / (1.0 + (D / D0) ** (2 * n))
    return H


# Show BLPF for different orders (fixed D0=30)
D0 = 30
orders = [1, 2, 5, 20]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, n in enumerate(orders):
    H = butterworth_lowpass_filter(P, Q, D0, n)
    axes[i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f'BLPF, $D_0$={D0}, n={n}', fontsize=12)
    axes[i].axis('off')

plt.suptitle('Butterworth Lowpass Filter: Increasing Order Sharpens the Transition',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Radial cross-section of BLPF for different orders
D = create_distance_matrix(P, Q)
center_u, center_v = P // 2, Q // 2
radial_distances = D[center_u, center_v:]

fig, ax = plt.subplots(figsize=(10, 6))

D0 = 30
for n in [1, 2, 5, 20]:
    H = butterworth_lowpass_filter(P, Q, D0, n)
    profile = H[center_u, center_v:]
    ax.plot(radial_distances, profile, linewidth=2, label=f'n = {n}')

# Also show ILPF for reference
H_ideal = ideal_lowpass_filter(P, Q, D0)
profile_ideal = H_ideal[center_u, center_v:]
ax.plot(radial_distances, profile_ideal, 'k--', linewidth=2, alpha=0.5, label='ILPF (n=inf)')

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.7, label='H = 0.5')
ax.axvline(x=D0, color='gray', linestyle=':', alpha=0.7)

ax.set_xlabel('Distance D(u,v) from Center', fontsize=12)
ax.set_ylabel('H(u,v)', fontsize=12)
ax.set_title(f'BLPF Radial Cross-Section ($D_0$={D0}): Effect of Filter Order n', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 80])
ax.set_ylim([-0.05, 1.1])
plt.tight_layout()
plt.show()

print(f"All Butterworth filters pass through H=0.5 at D={D0} (the cutoff frequency).")
print("Higher order n -> sharper transition -> approaches the Ideal filter.")

In [ ]:
# Apply BLPF with different orders
D0 = 30
orders = [1, 2, 5, 20]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, n in enumerate(orders):
    H = butterworth_lowpass_filter(P, Q, D0, n)
    filtered = apply_frequency_filter(test_img, H)
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'BLPF: n={n}', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(filtered, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Filtered (n={n})', fontsize=11)
    axes[1, i].axis('off')

plt.suptitle(f'Butterworth Lowpass Filter ($D_0$={D0}): Effect of Order n\n'
             'Low n = smooth (like GLPF) | High n = sharp (like ILPF, ringing appears)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Intensity profiles showing ringing for high-order Butterworth
row_idx = 40
D0 = 20

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test_img[row_idx, :], 'k-', linewidth=1, alpha=0.4, label='Original')

colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
for n, color in zip([1, 2, 5, 20], colors):
    H = butterworth_lowpass_filter(P, Q, D0, n)
    filtered = apply_frequency_filter(test_img, H)
    ax.plot(filtered[row_idx, :], color=color, linewidth=1.5, label=f'BLPF n={n}')

ax.set_xlabel('Column Index', fontsize=12)
ax.set_ylabel('Pixel Intensity', fontsize=12)
ax.set_title(f'Row {row_idx} Profile ($D_0$={D0}): Ringing Increases with Order n', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("n=1, n=2: Smooth transition, minimal or no ringing")
print("n=5, n=20: Sharper transition, visible ringing oscillations appear")

---
## 5. Comparison: ILPF vs GLPF vs BLPF

| Filter | Transfer Function | Transition | Ringing | Use Case |
|--------|-------------------|------------|---------|----------|
| **ILPF** | $H = \begin{cases} 1 & D \leq D_0 \\ 0 & D > D_0 \end{cases}$ | Sharp (discontinuous) | Severe | Theoretical only |
| **GLPF** | $H = e^{-D^2/(2D_0^2)}$ | Smooth (Gaussian) | None | Biomedical imaging |
| **BLPF** | $H = \frac{1}{1+(D/D_0)^{2n}}$ | Tunable (order $n$) | Depends on $n$ | General purpose |

### Relationship:
- BLPF with $n=1$ is similar to GLPF (smooth, no ringing)
- BLPF with $n \to \infty$ approaches ILPF (sharp, severe ringing)
- GLPF is the optimal choice when **no ringing** is acceptable

In [ ]:
# Side-by-side comparison: transfer functions
D0 = 30

H_ilpf = ideal_lowpass_filter(P, Q, D0)
H_glpf = gaussian_lowpass_filter(P, Q, D0)
H_blpf = butterworth_lowpass_filter(P, Q, D0, n=2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(H_ilpf, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'ILPF ($D_0$={D0})', fontsize=13)
axes[0].axis('off')

axes[1].imshow(H_glpf, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'GLPF ($D_0$={D0})', fontsize=13)
axes[1].axis('off')

axes[2].imshow(H_blpf, cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'BLPF ($D_0$={D0}, n=2)', fontsize=13)
axes[2].axis('off')

plt.suptitle('Transfer Functions: ILPF vs GLPF vs BLPF', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Radial profiles of all three filters on one plot
D = create_distance_matrix(P, Q)
center_u, center_v = P // 2, Q // 2
radial_distances = D[center_u, center_v:]

D0 = 30
H_ilpf = ideal_lowpass_filter(P, Q, D0)
H_glpf = gaussian_lowpass_filter(P, Q, D0)
H_blpf_1 = butterworth_lowpass_filter(P, Q, D0, n=1)
H_blpf_2 = butterworth_lowpass_filter(P, Q, D0, n=2)
H_blpf_5 = butterworth_lowpass_filter(P, Q, D0, n=5)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(radial_distances, H_ilpf[center_u, center_v:], 'k--', linewidth=2.5, label='ILPF')
ax.plot(radial_distances, H_glpf[center_u, center_v:], 'b-', linewidth=2.5, label='GLPF')
ax.plot(radial_distances, H_blpf_1[center_u, center_v:], 'g-', linewidth=2, label='BLPF n=1')
ax.plot(radial_distances, H_blpf_2[center_u, center_v:], 'r-', linewidth=2, label='BLPF n=2')
ax.plot(radial_distances, H_blpf_5[center_u, center_v:], 'm-', linewidth=2, label='BLPF n=5')

ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=D0, color='gray', linestyle=':', alpha=0.5)
ax.annotate(f'$D_0$ = {D0}', xy=(D0, 0.02), fontsize=11, ha='center')

ax.set_xlabel('Distance D(u,v) from Center', fontsize=12)
ax.set_ylabel('H(u,v)', fontsize=12)
ax.set_title(f'Comparison of Lowpass Filter Profiles ($D_0$={D0})', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 80])
ax.set_ylim([-0.05, 1.1])
plt.tight_layout()
plt.show()

In [ ]:
# Filtered results comparison: ILPF vs GLPF vs BLPF
D0 = 20

H_ilpf = ideal_lowpass_filter(P, Q, D0)
H_glpf = gaussian_lowpass_filter(P, Q, D0)
H_blpf = butterworth_lowpass_filter(P, Q, D0, n=2)

filt_ilpf = apply_frequency_filter(test_img, H_ilpf)
filt_glpf = apply_frequency_filter(test_img, H_glpf)
filt_blpf = apply_frequency_filter(test_img, H_blpf)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Top row: images
axes[0, 0].imshow(test_img, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(filt_ilpf, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title(f'ILPF ($D_0$={D0})', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(filt_glpf, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title(f'GLPF ($D_0$={D0})', fontsize=12)
axes[0, 2].axis('off')

axes[0, 3].imshow(filt_blpf, cmap='gray', vmin=0, vmax=255)
axes[0, 3].set_title(f'BLPF ($D_0$={D0}, n=2)', fontsize=12)
axes[0, 3].axis('off')

# Bottom row: intensity profiles along a row
row_idx = 40
for ax in axes[1, :]:
    ax.set_xlabel('Column', fontsize=10)
    ax.set_ylabel('Intensity', fontsize=10)
    ax.grid(True, alpha=0.3)

axes[1, 0].plot(test_img[row_idx, :], 'k-', linewidth=1.5)
axes[1, 0].set_title(f'Original Row {row_idx}', fontsize=11)

axes[1, 1].plot(filt_ilpf[row_idx, :], 'r-', linewidth=1.5)
axes[1, 1].set_title('ILPF: Ringing!', fontsize=11)

axes[1, 2].plot(filt_glpf[row_idx, :], 'b-', linewidth=1.5)
axes[1, 2].set_title('GLPF: Smooth', fontsize=11)

axes[1, 3].plot(filt_blpf[row_idx, :], 'g-', linewidth=1.5)
axes[1, 3].set_title('BLPF n=2: Mild', fontsize=11)

plt.suptitle(f'Comparison of Lowpass Filters ($D_0$={D0}): Images and Intensity Profiles',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Effect of Cutoff Frequency $D_0$

The cutoff frequency $D_0$ controls **how much blurring** is applied:

- **Small $D_0$** (e.g., 5-10): Only the lowest frequencies pass. Severe blurring, most detail lost.
- **Medium $D_0$** (e.g., 30-50): Moderate blurring, edges softened but still recognizable.
- **Large $D_0$** (e.g., 80-120): Mild blurring, fine noise removed but most features preserved.

### Relationship to Image Content:
- $D_0$ determines the **radius of the passband** in the frequency domain
- Everything outside this radius is attenuated (how much depends on filter type)
- In biomedical imaging, choosing the right $D_0$ is critical: too small loses diagnostic detail, too large does not remove enough noise

In [ ]:
# Effect of D0 on Gaussian Lowpass Filter
D0_values = [5, 10, 20, 40, 80, 120]

fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for i, D0 in enumerate(D0_values):
    H = gaussian_lowpass_filter(P, Q, D0)
    filtered = apply_frequency_filter(test_img, H)
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'$D_0$={D0}', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(filtered, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Result', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Filter H(u,v)', fontsize=12)
axes[1, 0].set_ylabel('Filtered Image', fontsize=12)

plt.suptitle('Effect of Cutoff Frequency $D_0$ on GLPF Smoothing\n'
             'Small $D_0$ = heavy blur | Large $D_0$ = mild blur',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Effect of D0 on Butterworth Lowpass Filter (n=2)
D0_values = [5, 10, 20, 40, 80, 120]

fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for i, D0 in enumerate(D0_values):
    H = butterworth_lowpass_filter(P, Q, D0, n=2)
    filtered = apply_frequency_filter(test_img, H)
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'$D_0$={D0}', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(filtered, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Result', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Filter H(u,v)', fontsize=12)
axes[1, 0].set_ylabel('Filtered Image', fontsize=12)

plt.suptitle('Effect of Cutoff Frequency $D_0$ on BLPF (n=2) Smoothing',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative analysis: how D0 affects image quality metrics
D0_range = np.arange(5, 128, 1)
mse_ilpf = []
mse_glpf = []
mse_blpf = []

for D0 in D0_range:
    filt_i = apply_frequency_filter(test_img, ideal_lowpass_filter(P, Q, D0))
    filt_g = apply_frequency_filter(test_img, gaussian_lowpass_filter(P, Q, D0))
    filt_b = apply_frequency_filter(test_img, butterworth_lowpass_filter(P, Q, D0, n=2))
    
    mse_ilpf.append(np.mean((test_img - filt_i) ** 2))
    mse_glpf.append(np.mean((test_img - filt_g) ** 2))
    mse_blpf.append(np.mean((test_img - filt_b) ** 2))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(D0_range, mse_ilpf, 'r-', linewidth=2, label='ILPF')
ax.plot(D0_range, mse_glpf, 'b-', linewidth=2, label='GLPF')
ax.plot(D0_range, mse_blpf, 'g-', linewidth=2, label='BLPF (n=2)')
ax.set_xlabel('Cutoff Frequency $D_0$', fontsize=12)
ax.set_ylabel('Mean Squared Error (MSE)', fontsize=12)
ax.set_title('Distortion vs Cutoff Frequency: Larger $D_0$ = Less Distortion', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([5, 127])
plt.tight_layout()
plt.show()

print("As D0 increases, the filter passes more frequencies and the image")
print("is less distorted (MSE decreases toward 0).")

---
## 7. Power Spectrum Analysis and Enclosed Energy

The **power spectrum** $P(u,v) = |F(u,v)|^2$ tells us how energy is distributed across frequencies.

### Total power:
$$P_{\text{total}} = \sum_u \sum_v |F(u,v)|^2$$

### Power enclosed within radius $D_0$:
$$P(D_0) = \sum_{D(u,v) \leq D_0} |F(u,v)|^2$$

### Fraction of total power:
$$\alpha(D_0) = \frac{P(D_0)}{P_{\text{total}}} \times 100\%$$

This tells us **how much of the image energy** is preserved by a lowpass filter with cutoff $D_0$.

For most images, a **large fraction** of total power is concentrated at **low frequencies** near the center.

In [ ]:
# Compute power spectrum and enclosed energy
F_centered = np.fft.fftshift(np.fft.fft2(test_img))
power_spectrum = np.abs(F_centered) ** 2
total_power = np.sum(power_spectrum)

D = create_distance_matrix(P, Q)

# Compute enclosed power for each radius
max_radius = int(np.max(D))
radii = np.arange(1, max_radius + 1)
enclosed_power = np.zeros(len(radii))

for i, r in enumerate(radii):
    mask = D <= r
    enclosed_power[i] = np.sum(power_spectrum[mask])

fraction = (enclosed_power / total_power) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Power spectrum image
axes[0].imshow(np.log1p(power_spectrum), cmap='hot')
axes[0].set_title('Power Spectrum (log scale)', fontsize=12)
axes[0].axis('off')

# Draw D0 circles
theta = np.linspace(0, 2 * np.pi, 100)
for D0, color in zip([10, 30, 60, 100], ['cyan', 'lime', 'yellow', 'white']):
    cx, cy = P // 2, Q // 2
    circle_x = cx + D0 * np.cos(theta)
    circle_y = cy + D0 * np.sin(theta)
    axes[0].plot(circle_y, circle_x, color=color, linewidth=1.5, label=f'$D_0$={D0}')
axes[0].legend(loc='upper right', fontsize=9)

# Enclosed power curve
axes[1].plot(radii, fraction, 'b-', linewidth=2)

# Mark specific D0 values
for D0 in [10, 30, 60, 100]:
    idx = D0 - 1
    axes[1].plot(D0, fraction[idx], 'ro', markersize=8)
    axes[1].annotate(f'$D_0$={D0}: {fraction[idx]:.1f}%',
                     xy=(D0, fraction[idx]),
                     xytext=(D0 + 8, fraction[idx] - 5),
                     fontsize=10, arrowprops=dict(arrowstyle='->', color='red'))

axes[1].set_xlabel('Radius $D_0$', fontsize=12)
axes[1].set_ylabel('Enclosed Power (%)', fontsize=12)
axes[1].set_title('Percentage of Total Power Within Radius $D_0$', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 130])
axes[1].set_ylim([80, 101])

plt.suptitle('Power Spectrum Analysis: Most Energy is at Low Frequencies',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Show the relationship between enclosed power and visual quality
D0_values = [5, 10, 20, 40, 80]

fig, axes = plt.subplots(2, len(D0_values), figsize=(18, 7))

for i, D0 in enumerate(D0_values):
    H = gaussian_lowpass_filter(P, Q, D0)
    filtered = apply_frequency_filter(test_img, H)
    
    # Power enclosed by this D0
    mask = D <= D0
    pct = np.sum(power_spectrum[mask]) / total_power * 100
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'$D_0$={D0}', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(filtered, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'{pct:.1f}% power', fontsize=11)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('GLPF H(u,v)', fontsize=12)
axes[1, 0].set_ylabel('Filtered Image', fontsize=12)

plt.suptitle('Enclosed Power vs Visual Quality (GLPF)\n'
             'Even a small circle encloses most of the power (image energy)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key insight: Most image energy is concentrated at low frequencies.")
print("Even D0=10 captures the majority of total power, but fine details are lost.")
print("The remaining few percent contain edges, texture, and noise.")

In [ ]:
# Demonstrate on a more complex synthetic biomedical image
def create_biomedical_phantom(size=256):
    """
    Create a more realistic biomedical phantom image:
    simulated cross-section with tissue regions, vessels, and noise.
    """
    img = np.ones((size, size), dtype=np.float64) * 30  # dark background
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    
    # Outer body contour (large ellipse)
    body = ((X - cx) / 100) ** 2 + ((Y - cy) / 80) ** 2 <= 1
    img[body] = 120
    
    # Organ 1 (brighter region)
    organ1 = ((X - cx + 30) / 35) ** 2 + ((Y - cy + 10) / 45) ** 2 <= 1
    img[organ1] = 170
    
    # Organ 2 (darker region)
    organ2 = ((X - cx - 35) / 25) ** 2 + ((Y - cy - 15) / 30) ** 2 <= 1
    img[organ2] = 80
    
    # Small bright spots (simulating calcifications)
    for (sx, sy, sr) in [(cx-20, cy+30, 5), (cx+40, cy-25, 4), (cx-50, cy-20, 3)]:
        spot = (X - sx) ** 2 + (Y - sy) ** 2 <= sr ** 2
        img[spot] = 240
    
    # Add Gaussian noise
    np.random.seed(123)
    img += np.random.normal(0, 8, img.shape)
    img = np.clip(img, 0, 255)
    
    return img

phantom = create_biomedical_phantom(256)

# Apply all three filters
D0 = 25
H_ilpf = ideal_lowpass_filter(P, Q, D0)
H_glpf = gaussian_lowpass_filter(P, Q, D0)
H_blpf = butterworth_lowpass_filter(P, Q, D0, n=2)

filt_ilpf = apply_frequency_filter(phantom, H_ilpf)
filt_glpf = apply_frequency_filter(phantom, H_glpf)
filt_blpf = apply_frequency_filter(phantom, H_blpf)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(phantom, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Biomedical Phantom\n(with noise)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(filt_ilpf, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'ILPF ($D_0$={D0})\nRinging artifacts', fontsize=12)
axes[1].axis('off')

axes[2].imshow(filt_glpf, cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'GLPF ($D_0$={D0})\nSmooth, no artifacts', fontsize=12)
axes[2].axis('off')

axes[3].imshow(filt_blpf, cmap='gray', vmin=0, vmax=255)
axes[3].set_title(f'BLPF ($D_0$={D0}, n=2)\nGood compromise', fontsize=12)
axes[3].axis('off')

plt.suptitle('Lowpass Filtering a Biomedical Phantom Image',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Clinical relevance:")
print("- GLPF and low-order BLPF are preferred in medical imaging")
print("- Ringing from ILPF can create false structures that mimic pathology")
print("- The choice of D0 balances noise reduction vs preservation of diagnostic features")

---
## Summary

What we learned:

1. **Lowpass filters** attenuate high frequencies and pass low frequencies, resulting in image **smoothing (blurring)**
2. **Ideal Lowpass Filter (ILPF)**: $H = 1$ if $D \leq D_0$, else $H = 0$. Sharp cutoff causes severe **ringing artifacts** (Gibbs phenomenon). Not used in practice.
3. **Gaussian Lowpass Filter (GLPF)**: $H = e^{-D^2/(2D_0^2)}$. Smooth transition produces **no ringing**. Optimal for biomedical applications.
4. **Butterworth Lowpass Filter (BLPF)**: $H = \frac{1}{1+(D/D_0)^{2n}}$. Order $n$ controls the sharpness of transition, from smooth ($n=1$, like GLPF) to sharp ($n \to \infty$, like ILPF).
5. **Cutoff frequency $D_0$** controls the amount of blurring: smaller $D_0$ = more blur, larger $D_0$ = less blur.
6. **Power spectrum analysis** shows that most image energy is concentrated at low frequencies. Even a small $D_0$ captures the majority of total power.
7. In **biomedical imaging**, GLPF and low-order BLPF are preferred because ringing artifacts from ILPF can create false structures that mimic pathology.